# Module detection (R) - CEMiTool

In [ ]:
library(DESeq2)
library(CEMiTool)
library(ggplot2)

## A. Load in cell type specific dataset specifically

In [ ]:
counts <- read.delim("MTG_cemitool_input/L2-3_IT_counts.tsv", row.names=1, check.names=FALSE)
annot  <- read.delim("MTG_cemitool_input/L2-3_IT_annot.tsv")

all(colnames(counts) == annot$SampleName)   # should be TRUE

In [ ]:
dim(counts)          # ~14324 genes × 43 donors
counts[1:3, 1:3]     # numeric counts, genes as rownames
head(annot)          # SampleName, Class columns

In [ ]:
dim(counts)
counts[1:3, 1:3]

## B. Run CEMiTool

In [ ]:
#### VST normalize counts matrix
expr <- as.data.frame(vst(as.matrix(counts), blind=TRUE))

#### RUN CEMiTool 
cem  <- cemitool(expr, annot, filter=TRUE, apply_vst=FALSE, verbose=TRUE)
nmodules(cem)

### Save modules for select cell types with correct name 

In [ ]:
mods <- module_genes(cem)
mod_list <- split(mods$genes, mods$modules)
mod_list <- mod_list[names(mod_list) != "Not.Correlated"]
gmt <- sapply(names(mod_list), function(m) paste(c(m, "NA", mod_list[[m]]), collapse="\t"))
writeLines(gmt, "MTG_cemitool_input/L2-3_IT_cemitool_modules.gmt")

In [ ]:
#### Uncover
nmodules(cem)                    # 6
head(module_genes(cem))          # gene-to-module assignments
mod_names <- unique(module_genes(cem)$modules)

real_modules <- setdiff(mod_names, "Not.Correlated")
print(real_modules)

In [ ]:
print(colnames(annot))

#### ---add MMSE info  (might not apply to you)---

In [ ]:
# strip the _celltype suffix to recover bare donor IDs
donor_id <- sub("_L2/3 IT$", "", annot$SampleName)

# read the individual metadata (same file behind your donor clustering)
meta <- read.csv("/tscc/nfs/home/aopatel/synapse_meta_NEW/SEA-AD_individual_metadata.csv",
                 check.names = FALSE)
print(colnames(meta))

In [ ]:
id_col   <- "individualID"          # <- adjust to what colnames(meta) shows
mmse_col <- "MMSE score"   # <- adjust

annot$MMSE <- as.numeric(meta[[mmse_col]][ match(donor_id, meta[[id_col]]) ])

# sanity check
print(data.frame(donor = donor_id, Class = annot$Class, MMSE = annot$MMSE))
cat("unmatched:", sum(is.na(annot$MMSE)), "of", nrow(annot), "\n")

## C. Eigengene correlation analysis

In [ ]:
eig <- mod_summary(cem, method = "eigengene")   # modules × donors, rows M1..M5

# eig currently has a 'modules' column + 43 donor columns
rownames(eig) <- eig$modules
eig$modules   <- NULL                    # now 6 x 43, pure numeric
eig <- as.matrix(eig)

# verify donor columns now align with annot
cat("eig dims:", paste(dim(eig), collapse=" x "), "\n")
print(all(colnames(eig) == annot$SampleName))    # should be TRUE now

In [ ]:
#### You can do this with diagnosis or primary condition if MMSE is unavailable

mmse <- annot$MMSE
mmse[mmse > 30] <- NA          # 33 is impossible -> treat as missing

mods <- setdiff(rownames(eig), "Not.Correlated")
res  <- data.frame(module=character(), rho=numeric(),
                   p=numeric(), n=integer(), stringsAsFactors=FALSE)

for (m in mods) {
  v  <- as.numeric(eig[m, ])
  ok <- !is.na(mmse) & !is.na(v)
  ct <- suppressWarnings(cor.test(v[ok], mmse[ok], method = "spearman"))
  res <- rbind(res, data.frame(module=m, rho=unname(ct$estimate),
                               p=ct$p.value, n=sum(ok)))
}

# BH-adjust across the modules tested
res$padj <- p.adjust(res$p, method = "BH")
res <- res[order(res$p), ]

cat("\n=== Module eigengene vs MMSE (Spearman, BH-adjusted) ===\n")
for (i in seq_len(nrow(res))) {
  cat(sprintf("%-4s rho=%+.2f  p=%.4f  padj=%.4f  n=%d  %s\n",
              res$module[i], res$rho[i], res$p[i], res$padj[i], res$n[i],
              ifelse(res$padj[i] < 0.05, "*", "")))
}

In [ ]:
all(colnames(eig) == annot$SampleName)   # should be TRUE

In [ ]:
#### Get hub genes for modules of most interest (this will be different for you)
m2_genes <- module_genes(cem, module="M2")$genes
m4_genes <- module_genes(cem, module="M4")$genes
cat("M2:", length(m2_genes), "genes | M4:", length(m4_genes), "genes\n")

hubs <- get_hubs(cem, n=10)
cat("\nM2 hubs:\n"); print(hubs$M2)
cat("\nM4 hubs:\n"); print(hubs$M4)

In [ ]:
#### This is a different GSEA of modules not based directly on 
#### differential gene expression, just use it to gauge directionality
#### and modules that seem to change a lot accross conditions
show_plot(cem, "gsea")